Read each paper's publication text plus the paper's .raw filenames, extract allowed SDRF metadat, and turn that into a valid submission table. Goal is not perfect row reconstruction, it is to recover the correct set of metadata values per paper and column as cleanly as possible. The dataset gives training paper JSONs, gold SDRFs, GPT-made training extracts, test paper JSONs, and a sample submission format.

Test_PubText/ is the test stuff for inference.
Training_GPT_Extract/ is GPT-generated metadata extracts for the training papers made using the baseline prompt, these are useful as baseline / weak teacher / candidate source
Training_PubText/ is the training papers as /json publication text files. These are the inputs we learn patterns from.
Training_SDRFs/ is the gold SDRF annotations fro the training set. This is the real target we compare against locally.
BaselinePrompt.txt is the official baseline extraction prompt, it defines what metadata categories are allowed, what manuscript sections to read, how to use .raw filenames, how to assign metadata per file, and what the output format should look like.

In [ ]:
from pathlib import Path
import json
import re
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel
from sklearn.cluster import AgglomerativeClustering

DATA_DIR = Path("Data")
TRAIN_PUB_DIR = DATA_DIR / "Training_PubText"
TRAIN_SDRF_DIR = DATA_DIR / "Training_SDRFs"
TEST_PUB_DIR = DATA_DIR / "Test_PubText"
SAMPLE_SUB_PATH = DATA_DIR / "SampleSubmission.csv"

DEFAULT_FILL = "Not Applicable"
NA_VALUES = {"", "nan", "none", "not applicable", "not available"}

sample_sub = pd.read_csv(SAMPLE_SUB_PATH)
SUB_COLUMNS = sample_sub.columns.tolist()

def load_sdrf(path: str | Path) -> pd.DataFrame:
    """
    Robust SDRF loader that also repairs files that get read as one giant column.
    """
    path = Path(path)
    df = pd.read_csv(path)

    if df.shape[1] == 1:
        first_col = str(df.columns[0])
        if "," in first_col:
            raw = pd.read_csv(path, header=None)
            split_rows = raw.iloc[:, 0].astype(str).str.split(",", expand=True)
            header = split_rows.iloc[0].tolist()
            data = split_rows.iloc[1:].reset_index(drop=True)
            data.columns = header
            df = data

    return df

def load_pub_json(pxd: str, split: str = "train") -> dict:
    folder = TRAIN_PUB_DIR if split == "train" else TEST_PUB_DIR
    path = folder / f"{pxd}_PubText.json"
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def load_gold_sdrf(pxd: str) -> pd.DataFrame:
    path = TRAIN_SDRF_DIR / f"Harmonized_{pxd}.csv"
    return load_sdrf(path)

def clean_text(x) -> str:
    if x is None:
        return ""
    return re.sub(r"\s+", " ", str(x)).strip()

def get_section_text(pub_json: dict, key: str) -> str:
    val = pub_json.get(key, "")
    if isinstance(val, str):
        return clean_text(val)
    if isinstance(val, list):
        return clean_text(" ".join(str(v) for v in val))
    if isinstance(val, dict):
        return clean_text(" ".join(f"{k}: {v}" for k, v in val.items()))
    return clean_text(val)

def get_core_text(pub_json: dict) -> Dict[str, str]:
    title = get_section_text(pub_json, "TITLE")
    abstract = get_section_text(pub_json, "ABSTRACT")
    methods = get_section_text(pub_json, "METHODS")
    core = "\n\n".join([x for x in [title, abstract, methods] if x])
    return {
        "title": title,
        "abstract": abstract,
        "methods": methods,
        "core": core,
    }

def extract_raw_files(pub_json: dict) -> List[str]:
    raw = pub_json.get("Raw Data Files", [])
    out = []

    if isinstance(raw, list):
        out = [clean_text(x) for x in raw if clean_text(x)]
    elif isinstance(raw, str):
        out = re.findall(r"[\w\-.]+\.(?:raw|RAW)", raw)
    else:
        out = []

    return list(dict.fromkeys(out))

def make_blank_row() -> Dict[str, str]:
    return {col: DEFAULT_FILL for col in SUB_COLUMNS}

def assign_if_present(row: dict, key: str, value):
    if key in row and value is not None and str(value).strip() != "":
        row[key] = str(value).strip()

def is_real_value(x) -> bool:
    return str(x).strip().lower() not in NA_VALUES

def unique_real_values_in_order(df: pd.DataFrame, col: str) -> List[str]:
    if col not in df.columns:
        return []

    vals = []
    seen = set()
    for v in df[col].dropna().astype(str).tolist():
        s = v.strip()
        if not is_real_value(s):
            continue
        k = s.lower()
        if k not in seen:
            seen.add(k)
            vals.append(s)
    return vals

TRAIN_PXDS = sorted([
    p.name.replace("_PubText.json", "")
    for p in TRAIN_PUB_DIR.glob("PXD*_PubText.json")
])

TEST_PXDS = sorted([
    p.name.replace("_PubText.json", "")
    for p in TEST_PUB_DIR.glob("PXD*_PubText.json")
])

print("Train PXDs:", len(TRAIN_PXDS))
print("Test PXDs :", len(TEST_PXDS))
print("Submission columns:", len(SUB_COLUMNS))

Train PXDs: 102
Test PXDs : 15
Submission columns: 81


In [ ]:
def norm_match(x: str) -> str:
    return re.sub(r"\s+", " ", str(x)).strip(" \t\r\n,.;:")

def find_all_patterns(text: str, patterns: List[str]) -> List[str]:
    vals = []
    seen = set()

    for pat in patterns:
        for m in re.finditer(pat, text, flags=re.IGNORECASE):
            v = norm_match(m.group(0))
            k = v.lower()
            if k not in seen:
                seen.add(k)
                vals.append(v)

    return vals

def pick_single(text: str, patterns: List[str]):
    vals = find_all_patterns(text, patterns)
    return vals[0] if len(vals) == 1 else None

def find_group1_single(text: str, patterns: List[str]):
    vals = []
    seen = set()

    for pat in patterns:
        for m in re.finditer(pat, text, flags=re.IGNORECASE):
            if m.lastindex is None:
                v = norm_match(m.group(0))
            else:
                v = norm_match(m.group(1))
            k = v.lower()
            if k not in seen:
                seen.add(k)
                vals.append(v)

    return vals[0] if len(vals) == 1 else None

def canonical_label(label_val):
    if label_val is None:
        return None
    x = str(label_val).lower()
    if "label free" in x:
        return "label free sample"
    return label_val

ORGANISM_PATTERNS = [
    r"\bplasmodium falciparum\b",
    r"\bhomo sapiens\b",
    r"\bmus musculus\b",
    r"\brattus norvegicus\b",
    r"\bsaccharomyces cerevisiae\b",
    r"\bescherichia coli\b",
    r"\barabidopsis thaliana\b",
    r"\bdrosophila melanogaster\b",
    r"\bcaenorhabditis elegans\b",
    r"\bdanio rerio\b",
]

ORGANISM_PART_PATTERNS = [
    r"\bhuman erythrocytes\b",
    r"\berythrocytes\b",
    r"\bplasma\b",
    r"\bserum\b",
    r"\bliver\b",
    r"\bbrain\b",
    r"\bheart\b",
    r"\blung\b",
    r"\bkidney\b",
    r"\bcolon\b",
    r"\bprostate\b",
    r"\bbreast\b",
]

CELLTYPE_PATTERNS = [
    r"\bschizont\b",
    r"\bneuron(?:s)?\b",
    r"\bfibroblast(?:s)?\b",
    r"\bhepatocyte(?:s)?\b",
    r"\blymphocyte(?:s)?\b",
    r"\bmacrophage(?:s)?\b",
    r"\bstem cell(?:s)?\b",
    r"\bT cell(?:s)?\b",
    r"\bB cell(?:s)?\b",
]

CELLLINE_PATTERNS = [
    r"\bHEK293T\b",
    r"\bHEK293\b",
    r"\bU2OS\b",
    r"\bHeLa\b",
    r"\bMCF-7\b",
    r"\bA549\b",
    r"\bK562\b",
    r"\bJurkat\b",
]

LABEL_PATTERNS = [
    r"\blabel[- ]free(?: sample)?\b",
    r"\bTMTpro[- ]?\d+[NC]?\b",
    r"\bTMT[- ]?\d+[NC]?\b",
    r"\biTRAQ[- ]?\d+\b",
    r"\bSILAC(?: [A-Za-z]+)?\b",
]

FRAGMENTATION_PATTERNS = [
    r"\bHCD\b",
    r"\bCID\b",
    r"\bETD\b",
    r"\bECD\b",
    r"\bPQD\b",
]

ACQUISITION_PATTERNS = [
    r"\bDDA\b",
    r"\bDIA\b",
    r"\bPRM\b",
    r"\bSRM\b",
    r"\bMRM\b",
]

ENRICHMENT_PATTERNS = [
    r"\bIMAC\b",
    r"\bTiO2\b",
    r"\btitanium dioxide\b",
    r"\banti-?phosphotyrosine\b",
    r"\bSCX\b",
]

IONIZATION_PATTERNS = [
    r"\bnanoESI\b",
    r"\bESI\b",
    r"\bMALDI\b",
]

MS2_ANALYZER_PATTERNS = [
    r"\borbitrap\b",
    r"\bion trap\b",
    r"\bquadrupole\b",
    r"\bTOF\b",
    r"\bFT-?ICR\b",
]

ALKYLATION_PATTERNS = [
    r"\biodoacetamide\b",
    r"\bIAA\b",
    r"\bNEM\b",
    r"\bchloroacetamide\b",
    r"\bCAA\b",
]

REDUCTION_PATTERNS = [
    r"\bDTT\b",
    r"\bTCEP\b",
    r"\bβ-mercaptoethanol\b",
    r"\bbeta-mercaptoethanol\b",
]

FLOW_RATE_PATTERNS = [
    r"(?:flow rate(?: of)?\s*)(\d+(?:\.\d+)?\s?(?:nL/min|uL/min|µL/min|mL/min))",
    r"(\d+(?:\.\d+)?\s?(?:nL/min|uL/min|µL/min|mL/min)\s+flow rate)",
]

GRADIENT_TIME_PATTERNS = [
    r"(?:gradient(?: length| time)?(?: of)?\s*)(\d+(?:\.\d+)?\s?(?:min|minutes))",
    r"(\d+(?:\.\d+)?\s?(?:min|minutes)\s+gradient)",
]

PRECURSOR_TOL_PATTERNS = [
    r"(?:precursor(?: mass)? tolerance(?: of)?\s*)(\d+(?:\.\d+)?\s?(?:ppm|Da))",
    r"(\d+(?:\.\d+)?\s?(?:ppm|Da)\s+precursor(?: mass)? tolerance)",
]

FRAGMENT_TOL_PATTERNS = [
    r"(?:fragment(?: mass)? tolerance(?: of)?\s*)(\d+(?:\.\d+)?\s?(?:ppm|Da))",
    r"(\d+(?:\.\d+)?\s?(?:ppm|Da)\s+fragment(?: mass)? tolerance)",
]

MISSED_CLEAVAGES_PATTERNS = [
    r"(?:up to |maximum of |max(?:imum)? )?\d+\s+missed cleavages?\b",
]

def extract_structured_metadata(pub_json: dict) -> Dict[str, str]:
    sec = get_core_text(pub_json)
    text = sec["core"]

    out = {}

    out["Comment[FlowRateChromatogram]"] = find_group1_single(text, FLOW_RATE_PATTERNS)
    out["Comment[GradientTime]"] = find_group1_single(text, GRADIENT_TIME_PATTERNS)
    out["Comment[PrecursorMassTolerance]"] = find_group1_single(text, PRECURSOR_TOL_PATTERNS)
    out["Comment[FragmentMassTolerance]"] = find_group1_single(text, FRAGMENT_TOL_PATTERNS)
    out["Comment[NumberOfMissedCleavages]"] = pick_single(text, MISSED_CLEAVAGES_PATTERNS)

    explicit_cols = {
        "Characteristics[Organism]": ORGANISM_PATTERNS,
        "Characteristics[OrganismPart]": ORGANISM_PART_PATTERNS,
        "Characteristics[CellType]": CELLTYPE_PATTERNS,
        "Characteristics[CellLine]": CELLLINE_PATTERNS,
        "Characteristics[Label]": LABEL_PATTERNS,
        "Comment[FragmentationMethod]": FRAGMENTATION_PATTERNS,
        "Comment[AcquisitionMethod]": ACQUISITION_PATTERNS,
        "Comment[EnrichmentMethod]": ENRICHMENT_PATTERNS,
        "Comment[IonizationType]": IONIZATION_PATTERNS,
        "Comment[MS2MassAnalyzer]": MS2_ANALYZER_PATTERNS,
        "Characteristics[AlkylationReagent]": ALKYLATION_PATTERNS,
        "Characteristics[ReductionReagent]": REDUCTION_PATTERNS,
    }

    for col, pats in explicit_cols.items():
        val = pick_single(text, pats)
        if col == "Characteristics[Label]":
            val = canonical_label(val)
        out[col] = val

    return out

def parse_filename_metadata(raw_file: str) -> Dict[str, str]:
    name = Path(str(raw_file)).name
    stem = name.rsplit(".", 1)[0]

    out = {}

    m = re.search(r"(?i)(?:^|[_\-.])(F|FRAC|FRACTION)(\d{1,3})(?:$|[_\-.])", stem)
    if m:
        out["Comment[FractionIdentifier]"] = m.group(2).lstrip("0") or "0"

    m = re.search(r"(?i)(?:^|[_\-.])(?:BR|BIOREP|BIOLOGICALREP(?:LICATE)?)(\d{1,3})(?:$|[_\-.])", stem)
    if m:
        out["Characteristics[BiologicalReplicate]"] = m.group(1).lstrip("0") or "0"

    m = re.search(r"(?i)(?:^|[_\-.])(\d+(?:h|hr|hrs|hour|hours|d|day|days|w|week|weeks))(?:$|[_\-.])", stem)
    if m:
        out["Characteristics[Time]"] = m.group(1)

    m = re.search(r"(?i)(TMTpro[- ]?\d+[NC]?|TMT[- ]?\d+[NC]?|iTRAQ[- ]?\d+|SILAC(?:[-_ ]?[A-Za-z]+)?)", stem)
    if m:
        out["Characteristics[Label]"] = canonical_label(m.group(1))

    if re.search(r"(?i)(?:^|[_\-.])(ctrl|control|vehicle|untreated)(?:$|[_\-.])", stem):
        out["Characteristics[Treatment]"] = "control"
    elif re.search(r"(?i)(?:^|[_\-.])(ko|knockout)(?:$|[_\-.])", stem):
        out["Characteristics[GeneticModification]"] = "knockout"
    elif re.search(r"(?i)(?:^|[_\-.])(wt|wildtype)(?:$|[_\-.])", stem):
        out["Characteristics[Genotype]"] = "wild type"

    return out

In [ ]:
VALUE_BANK_COLS = [
    "Characteristics[Organism]",
    "Characteristics[OrganismPart]",
    "Characteristics[CellType]",
    "Characteristics[CellLine]",
    "Characteristics[Label]",
    "Characteristics[BiologicalReplicate]",
    "Comment[Instrument]",
    "Comment[FragmentationMethod]",
    "Characteristics[CleavageAgent]",
    "Comment[AcquisitionMethod]",
    "Comment[EnrichmentMethod]",
    "Comment[IonizationType]",
    "Comment[MS2MassAnalyzer]",
    "Characteristics[AlkylationReagent]",
    "Characteristics[ReductionReagent]",
    "Characteristics[MaterialType]",
    "Comment[PrecursorMassTolerance]",
    "Comment[FragmentMassTolerance]",
    "Comment[FlowRateChromatogram]",
    "Comment[GradientTime]",
    "Comment[NumberOfMissedCleavages]",
    "Characteristics[Modification]",
    "Characteristics[Modification].1",
    "Characteristics[Modification].2",
    "Characteristics[Modification].3",
    "Characteristics[Modification].4",
    "Characteristics[Modification].5",
    "Characteristics[Modification].6",
    "Characteristics[Modification].7",
]

value_bank_rows = []

for pxd in TRAIN_PXDS:
    pub_json = load_pub_json(pxd, split="train")
    gold = load_gold_sdrf(pxd)
    sec = get_core_text(pub_json)

    row = {
        "pxd": pxd,
        "core_text": sec["core"],
        "n_raw_files": len(extract_raw_files(pub_json)),
    }

    for col in VALUE_BANK_COLS:
        row[col] = unique_real_values_in_order(gold, col)

    value_bank_rows.append(row)

study_value_bank_df = pd.DataFrame(value_bank_rows)

valuebank_vectorizer = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3, 5),
    min_df=1,
    lowercase=True,
    strip_accents="unicode",
)

valuebank_X = valuebank_vectorizer.fit_transform(
    study_value_bank_df["core_text"].fillna("").tolist()
)

print("study_value_bank_df shape:", study_value_bank_df.shape)
print("valuebank_X shape:", valuebank_X.shape)
display(study_value_bank_df[["pxd", "n_raw_files"]].head())

study_value_bank_df shape: (102, 32)
valuebank_X shape: (102, 141138)


,pxd,n_raw_files
0,PXD000070,6
1,PXD000534,16
2,PXD000561,2212
3,PXD000651,37
4,PXD000652,46


In [ ]:
SINGLE_VALUE_COLS_V3 = [
    "Characteristics[Organism]",
    "Characteristics[OrganismPart]",
    "Characteristics[CellType]",
    "Characteristics[CellLine]",
    "Comment[Instrument]",
    "Comment[FragmentationMethod]",
    "Characteristics[CleavageAgent]",
    "Comment[AcquisitionMethod]",
    "Comment[EnrichmentMethod]",
    "Comment[IonizationType]",
    "Comment[MS2MassAnalyzer]",
    "Characteristics[AlkylationReagent]",
    "Characteristics[ReductionReagent]",
    "Characteristics[MaterialType]",
    "Comment[PrecursorMassTolerance]",
    "Comment[FragmentMassTolerance]",
    "Comment[FlowRateChromatogram]",
    "Comment[GradientTime]",
    "Comment[NumberOfMissedCleavages]",
    "Characteristics[Modification]",
    "Characteristics[Modification].1",
    "Characteristics[Modification].2",
    "Characteristics[Modification].3",
    "Characteristics[Modification].4",
    "Characteristics[Modification].5",
    "Characteristics[Modification].6",
    "Characteristics[Modification].7",
]

MULTI_VALUE_COLS_V3 = {
    "Characteristics[Label]": {"max_values": 3, "min_share": 0.18},
    "Characteristics[BiologicalReplicate]": {"max_values": 8, "min_share": 0.12},
}

LOWER_THRESHOLD_COLS = {
    "Comment[Instrument]",
    "Characteristics[CleavageAgent]",
    "Characteristics[Modification]",
    "Characteristics[Modification].1",
    "Characteristics[Modification].2",
    "Characteristics[Modification].3",
    "Characteristics[Modification].4",
    "Characteristics[Modification].5",
    "Characteristics[Modification].6",
    "Characteristics[Modification].7",
    "Comment[FragmentationMethod]",
    "Comment[EnrichmentMethod]",
    "Comment[MS2MassAnalyzer]",
}

def get_neighbors_for_pub(pub_json: dict, exclude_pxd: str | None = None, top_k: int = 10, min_sim: float = 0.05) -> pd.DataFrame:
    sec = get_core_text(pub_json)
    q_text = sec["core"]

    qX = valuebank_vectorizer.transform([q_text])
    sims = linear_kernel(qX, valuebank_X).ravel()
    order = np.argsort(-sims)

    rows = []
    for idx in order:
        pxd = study_value_bank_df.iloc[idx]["pxd"]
        sim = float(sims[idx])

        if exclude_pxd is not None and pxd == exclude_pxd:
            continue
        if sim < min_sim:
            continue

        rows.append({
            "idx": idx,
            "pxd": pxd,
            "sim": sim,
        })

        if len(rows) >= top_k:
            break

    return pd.DataFrame(rows)

def vote_single_value(neighbors_df: pd.DataFrame, col: str):
    if len(neighbors_df) == 0:
        return None

    weights = {}
    support_weight = 0.0

    for _, nr in neighbors_df.iterrows():
        idx = int(nr["idx"])
        sim = float(nr["sim"])
        vals = study_value_bank_df.iloc[idx][col]

        if not isinstance(vals, list) or len(vals) == 0:
            continue

        support_weight += sim
        for v in vals:
            weights[v] = weights.get(v, 0.0) + sim

    if len(weights) == 0 or support_weight <= 0:
        return None

    ranked = sorted(weights.items(), key=lambda x: (-x[1], x[0]))
    best_val, best_w = ranked[0]
    best_share = best_w / support_weight

    min_share = 0.45
    if col in LOWER_THRESHOLD_COLS:
        min_share = 0.30

    return best_val if best_share >= min_share else None

def vote_multi_values(neighbors_df: pd.DataFrame, col: str, max_values: int = 3, min_share: float = 0.18) -> List[str]:
    if len(neighbors_df) == 0:
        return []

    weights = {}
    support_weight = 0.0

    for _, nr in neighbors_df.iterrows():
        idx = int(nr["idx"])
        sim = float(nr["sim"])
        vals = study_value_bank_df.iloc[idx][col]

        if not isinstance(vals, list) or len(vals) == 0:
            continue

        support_weight += sim
        for v in vals:
            weights[v] = weights.get(v, 0.0) + sim

    if len(weights) == 0 or support_weight <= 0:
        return []

    ranked = sorted(weights.items(), key=lambda x: (-x[1], x[0]))
    out = []

    for v, w in ranked:
        share = w / support_weight
        if share >= min_share:
            out.append(v)
        if len(out) >= max_values:
            break

    return out

def retrieve_neighbor_predictions_v3(pub_json: dict, exclude_pxd: str | None = None, top_k: int = 10, min_sim: float = 0.05):
    neighbors_df = get_neighbors_for_pub(pub_json, exclude_pxd=exclude_pxd, top_k=top_k, min_sim=min_sim)

    single_preds = {}
    for col in SINGLE_VALUE_COLS_V3:
        val = vote_single_value(neighbors_df, col)
        if val is not None:
            single_preds[col] = val

    multi_preds = {}
    for col, cfg in MULTI_VALUE_COLS_V3.items():
        vals = vote_multi_values(
            neighbors_df,
            col,
            max_values=cfg["max_values"],
            min_share=cfg["min_share"],
        )
        if len(vals) > 0:
            multi_preds[col] = vals

    return single_preds, multi_preds, neighbors_df

In [ ]:
def assign_round_robin(rows: List[dict], col: str, values: List[str], overwrite_if_default_only: bool = True):
    if len(rows) == 0 or len(values) == 0:
        return

    n = len(values)
    for i, row in enumerate(rows):
        if overwrite_if_default_only and row.get(col, DEFAULT_FILL) != DEFAULT_FILL:
            continue
        row[col] = values[i % n]

def merge_metadata_dicts(*dicts) -> Dict[str, str]:
    out = {}
    for d in dicts:
        for k, v in d.items():
            if v is None:
                continue
            s = str(v).strip()
            if s == "":
                continue
            out[k] = s
    return out

def baseline_predict_pxd_v3(pxd: str, split: str = "train", debug: bool = False) -> pd.DataFrame:
    pub_json = load_pub_json(pxd, split=split)
    raw_files = extract_raw_files(pub_json)

    exclude = pxd if split == "train" else None

    explicit_meta = extract_structured_metadata(pub_json)

    nn_single, nn_multi, nn_df = retrieve_neighbor_predictions_v3(
        pub_json,
        exclude_pxd=exclude,
        top_k=10,
        min_sim=0.05,
    )

    global_meta = merge_metadata_dicts(explicit_meta, nn_single)

    rows = []
    for i, raw_file in enumerate(raw_files, start=1):
        row = make_blank_row()

        if "ID" in row:
            row["ID"] = i
        if "PXD" in row:
            row["PXD"] = pxd
        if "Raw Data File" in row:
            row["Raw Data File"] = raw_file

        for k, v in global_meta.items():
            assign_if_present(row, k, v)

        file_meta = parse_filename_metadata(raw_file)
        for k, v in file_meta.items():
            assign_if_present(row, k, v)

        rows.append(row)

    if "Characteristics[Label]" in nn_multi:
        assign_round_robin(rows, "Characteristics[Label]", nn_multi["Characteristics[Label]"])

    if "Characteristics[BiologicalReplicate]" in nn_multi:
        rep_vals = nn_multi["Characteristics[BiologicalReplicate]"]

        def rep_key(x):
            m = re.search(r"\d+", str(x))
            return int(m.group(0)) if m else 10**9

        rep_vals = sorted(rep_vals, key=rep_key)
        rep_vals = rep_vals[:min(len(rep_vals), max(1, min(8, len(rows))))]

        assign_round_robin(rows, "Characteristics[BiologicalReplicate]", rep_vals)

    pred_df = pd.DataFrame(rows)

    for col in SUB_COLUMNS:
        if col not in pred_df.columns:
            pred_df[col] = DEFAULT_FILL

    pred_df = pred_df[SUB_COLUMNS]

    if debug:
        print("=== explicit_meta ===")
        for k, v in explicit_meta.items():
            if v is not None:
                print(k, "->", v)

        print("\n=== nn_single ===")
        for k, v in nn_single.items():
            print(k, "->", v)

        print("\n=== nn_multi ===")
        for k, v in nn_multi.items():
            print(k, "->", v)

        print("\n=== neighbors ===")
        display(nn_df)

    return pred_df

pred_v3_example = baseline_predict_pxd_v3(TRAIN_PXDS[0], split="train", debug=False)
print("Example prediction shape:", pred_v3_example.shape)
display(pred_v3_example.head())

Example prediction shape: (6, 81)


,ID,PXD,Raw Data File,Characteristics[Age],Characteristics[AlkylationReagent],Characteristics[AnatomicSiteTumor],Characteristics[AncestryCategory],Characteristics[BMI],Characteristics[Bait],Characteristics[BiologicalReplicate],...,FactorValue[Bait],FactorValue[CellPart],FactorValue[Compound],FactorValue[ConcentrationOfCompound].1,FactorValue[Disease],FactorValue[FractionIdentifier],FactorValue[GeneticModification],FactorValue[Temperature],FactorValue[Treatment],Usage
0,1,PXD000070,OTPf-IMACDDNL_2010Mar9-01.raw,Not Applicable,IAA,Not Applicable,Not Applicable,Not Applicable,Not Applicable,1,...,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable
1,2,PXD000070,OTPf-IMACDT2010Mar11-01.raw,Not Applicable,IAA,Not Applicable,Not Applicable,Not Applicable,Not Applicable,2,...,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable
2,3,PXD000070,OTPf-IMACDT2010Mar11-02.raw,Not Applicable,IAA,Not Applicable,Not Applicable,Not Applicable,Not Applicable,1,...,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable
3,4,PXD000070,OTPf-IMACDT2010Mar10-01.raw,Not Applicable,IAA,Not Applicable,Not Applicable,Not Applicable,Not Applicable,2,...,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable
4,5,PXD000070,OTPf-IMACDDNL_2010Mar9-02.raw,Not Applicable,IAA,Not Applicable,Not Applicable,Not Applicable,Not Applicable,1,...,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable


In [ ]:
import difflib

class ParticipantVisibleError(Exception):
    pass

def official_load_sdrf(sdrf_df: pd.DataFrame) -> Dict[str, Dict[str, List[str]]]:

    sdrf_dict: Dict[str, Dict[str, List[str]]] = {}
    for pxd, pxd_df in sdrf_df.groupby("PXD"):
        sdrf_dict[pxd] = {}
        for col in pxd_df.columns:
            if col in ["Raw Data File", "Usage", "PXD"]:
                continue

            uniq = pd.Series(pxd_df[col]).dropna().astype(str).unique().tolist()

            if uniq == ["Not Applicable"]:
                continue

            values: List[str] = []
            for v in uniq:
                if "NT=" in v:
                    parts = [r for r in v.split(";") if "NT=" in r]
                    values.append(parts[0].replace("NT=", "").strip() if parts else v.strip())
                else:
                    values.append(v.strip())

            if "." in col:
                col = col.split(".")[0].strip()

            if col in sdrf_dict[pxd]:
                sdrf_dict[pxd][col] += values
            else:
                sdrf_dict[pxd][col] = values

    return sdrf_dict

def _string_similarity(a: str, b: str) -> float:
    return difflib.SequenceMatcher(None, a or "", b or "").ratio()

def official_harmonize_and_evaluate(
    A: Dict[str, Dict[str, List[str]]],
    B: Dict[str, Dict[str, List[str]]],
    threshold: float = 0.80,
    CompleteAbsence: float = float("nan"),
) -> Tuple[Dict[str, Dict[str, List[int]]], Dict[str, Dict[str, List[int]]], pd.DataFrame]:

    from sklearn.metrics import precision_score, recall_score, f1_score

    eval_metrics = {
        "pxd": [],
        "AnnotationType": [],
        "precision": [],
        "recall": [],
        "f1": [],
        "jacc": [],
    }

    harmonized_A: Dict[str, Dict[str, List[int]]] = {}
    harmonized_B: Dict[str, Dict[str, List[int]]] = {}

    common_pubs = set(A) & set(B)
    for pub in common_pubs:
        harmonized_A[pub], harmonized_B[pub] = {}, {}

        for category in set(A[pub]):
            vals_A = A[pub][category]
            vals_B = B[pub].get(category, [])

            all_vals = vals_A + [v for v in vals_B if v not in vals_A]

            if len(vals_A) == 0 and len(vals_B) == 0:
                harmA, harmB = [], []

            elif len(all_vals) == 1:
                str2cid = {all_vals[0]: 0}
                harmA = [str2cid[s] for s in vals_A]
                harmB = [str2cid[s] for s in vals_B]

            else:
                N = len(all_vals)
                dist = np.zeros((N, N), dtype=float)

                for i in range(N):
                    for j in range(i + 1, N):
                        sim = _string_similarity(all_vals[i], all_vals[j])
                        d = 1.0 - sim
                        dist[i, j] = d
                        dist[j, i] = d

                clusterer = AgglomerativeClustering(
                    n_clusters=None,
                    metric="precomputed",
                    linkage="average",
                    distance_threshold=1.0 - threshold,
                )
                labels = clusterer.fit_predict(dist)
                str2cid = {s: int(labels[i]) for i, s in enumerate(all_vals)}
                harmA = [str2cid[s] for s in vals_A]
                harmB = [str2cid[s] for s in vals_B]

            harmonized_A[pub][category] = harmA
            harmonized_B[pub][category] = harmB

            uniq = sorted(set(harmA) | set(harmB))
            if not uniq:
                p = r = f = CompleteAbsence
                j = 1.0
            else:
                y_true = [1 if u in harmA else 0 for u in uniq]
                y_pred = [1 if u in harmB else 0 for u in uniq]

                p = precision_score(y_true, y_pred, average="macro", zero_division=0)
                r = recall_score(y_true, y_pred, average="macro", zero_division=0)
                f = f1_score(y_true, y_pred, average="macro", zero_division=0)

                setA, setB = set(harmA), set(harmB)
                j = 1.0 if (not setA and not setB) else len(setA & setB) / len(setA | setB)

            eval_metrics["pxd"].append(pub)
            eval_metrics["AnnotationType"].append(category)
            eval_metrics["precision"].append(p)
            eval_metrics["recall"].append(r)
            eval_metrics["f1"].append(f)
            eval_metrics["jacc"].append(j)

    return harmonized_A, harmonized_B, pd.DataFrame(eval_metrics)

def official_score(solution: pd.DataFrame, submission: pd.DataFrame, row_id_column_name: str = "ID") -> float:
    if row_id_column_name and row_id_column_name in solution.columns:
        solution = solution.drop(columns=[row_id_column_name])
    if row_id_column_name and row_id_column_name in submission.columns:
        submission = submission.drop(columns=[row_id_column_name])

    if "PXD" not in solution.columns or "PXD" not in submission.columns:
        raise ParticipantVisibleError("Both solution and submission must include a 'PXD' column.")

    sol = official_load_sdrf(solution)
    sub = official_load_sdrf(submission)
    _, _, eval_df = official_harmonize_and_evaluate(sol, sub, threshold=0.80)

    vals = eval_df["f1"].dropna()
    return float(vals.mean()) if not vals.empty else 0.0

In [ ]:
def build_train_solution_df(train_pxds: List[str]) -> pd.DataFrame:
    parts = []
    for pxd in train_pxds:
        gold = load_gold_sdrf(pxd).copy()

        for col in SUB_COLUMNS:
            if col not in gold.columns:
                gold[col] = DEFAULT_FILL

        gold = gold[SUB_COLUMNS]
        parts.append(gold)

    return pd.concat(parts, ignore_index=True)

def build_train_submission_df(train_pxds: List[str], predictor_fn) -> pd.DataFrame:
    parts = []
    failed = []

    for pxd in train_pxds:
        try:
            pred = predictor_fn(pxd, split="train").copy()

            for col in SUB_COLUMNS:
                if col not in pred.columns:
                    pred[col] = DEFAULT_FILL

            pred = pred[SUB_COLUMNS]
            parts.append(pred)

        except Exception as e:
            failed.append((pxd, str(e)))

    if failed:
        print("FAILED PXDs:")
        for x in failed[:20]:
            print(x)

    out = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame(columns=SUB_COLUMNS)

    if "ID" in out.columns:
        out["ID"] = np.arange(1, len(out) + 1)

    return out

def official_eval_breakdown(solution_df: pd.DataFrame, submission_df: pd.DataFrame, row_id_column_name: str = "ID"):
    sol = solution_df.copy()
    sub = submission_df.copy()

    if row_id_column_name and row_id_column_name in sol.columns:
        sol = sol.drop(columns=[row_id_column_name])
    if row_id_column_name and row_id_column_name in sub.columns:
        sub = sub.drop(columns=[row_id_column_name])

    sol_dict = official_load_sdrf(sol)
    sub_dict = official_load_sdrf(sub)

    _, _, eval_df = official_harmonize_and_evaluate(sol_dict, sub_dict, threshold=0.80)

    by_col = (
        eval_df.groupby("AnnotationType", as_index=False)[["precision", "recall", "f1", "jacc"]]
        .mean()
        .sort_values("f1", ascending=False)
        .reset_index(drop=True)
    )

    by_col["n_pxd"] = (
        eval_df.groupby("AnnotationType")["pxd"]
        .nunique()
        .reindex(by_col["AnnotationType"])
        .values
    )

    return eval_df, by_col

gold_train_df = build_train_solution_df(TRAIN_PXDS)
pred_train_v3_df = build_train_submission_df(TRAIN_PXDS, baseline_predict_pxd_v3)

train_score_v3 = official_score(
    solution=gold_train_df.copy(),
    submission=pred_train_v3_df.copy(),
    row_id_column_name="ID",
)

eval_df_v3, by_col_v3 = official_eval_breakdown(
    gold_train_df,
    pred_train_v3_df,
    row_id_column_name="ID",
)

print("V3 official train score:", train_score_v3)
display(by_col_v3.head(40))

V3 official train score: 0.3016393018714523


,AnnotationType,precision,recall,f1,jacc,n_pxd
0,Characteristics[AlkylationReagent],1.000000,1.000000,1.000000,1.000000,2
1,Characteristics[ReductionReagent],1.000000,1.000000,1.000000,1.000000,2
2,Characteristics[CleavageAgent],0.901961,0.885621,0.890289,0.898693,102
3,Comment[MS2MassAnalyzer],0.880952,0.869048,0.873016,0.880952,21
4,Characteristics[Organism],0.769608,0.767157,0.767974,0.769608,102
5,Comment[FragmentationMethod],0.663043,0.643116,0.648551,0.655797,46
6,Characteristics[BiologicalReplicate],0.405971,0.545333,0.433230,0.553638,97
7,Comment[PrecursorMassTolerance],0.418919,0.418919,0.418919,0.418919,74
8,Characteristics[Label],0.364841,0.423203,0.380923,0.415583,102
9,Comment[EnrichmentMethod],0.388889,0.305556,0.333333,0.388889,9


In [ ]:
sample_sub = pd.read_csv(SAMPLE_SUB_PATH).copy()

def fit_prediction_to_target_rows(pred_df: pd.DataFrame, target_n: int) -> pd.DataFrame:
    pred_df = pred_df.copy().reset_index(drop=True)

    if len(pred_df) == 0:
        out = pd.DataFrame([{col: DEFAULT_FILL for col in SUB_COLUMNS} for _ in range(target_n)])
        return out[SUB_COLUMNS]

    if len(pred_df) == target_n:
        return pred_df[SUB_COLUMNS].copy()

    if len(pred_df) < target_n:
        reps = int(np.ceil(target_n / len(pred_df)))
        out = pd.concat([pred_df] * reps, ignore_index=True).iloc[:target_n].copy()
        return out[SUB_COLUMNS]

    out = pred_df.iloc[:target_n].copy().reset_index(drop=True)
    return out[SUB_COLUMNS]

submission_parts = []

for pxd, template_df in sample_sub.groupby("PXD", sort=False):
    target_n = len(template_df)

    pred_df = baseline_predict_pxd_v3(pxd, split="test").copy()

    for col in SUB_COLUMNS:
        if col not in pred_df.columns:
            pred_df[col] = DEFAULT_FILL
    pred_df = pred_df[SUB_COLUMNS]

    pred_df = fit_prediction_to_target_rows(pred_df, target_n)

    if "ID" in pred_df.columns and "ID" in template_df.columns:
        pred_df["ID"] = template_df["ID"].values

    pred_df["PXD"] = template_df["PXD"].values

    submission_parts.append(pred_df)

submission_v3 = pd.concat(submission_parts, ignore_index=True)

for col in SUB_COLUMNS:
    if col not in submission_v3.columns:
        submission_v3[col] = DEFAULT_FILL
submission_v3 = submission_v3[SUB_COLUMNS]

print("submission_v3 shape:", submission_v3.shape)
print("sample_sub shape   :", sample_sub.shape)
print("unique IDs         :", submission_v3["ID"].nunique(), "out of", len(submission_v3))

count_check = (
    sample_sub.groupby("PXD").size().reset_index(name="sample_rows")
    .merge(submission_v3.groupby("PXD").size().reset_index(name="pred_rows"), on="PXD", how="left")
)
count_check["diff"] = count_check["pred_rows"] - count_check["sample_rows"]
display(count_check)

submission_v3.to_csv("submission_v3.csv", index=False)
print("saved -> submission_v3.csv")

submission_v3 shape: (1659, 81)
sample_sub shape   : (1659, 81)
unique IDs         : 1659 out of 1659


,PXD,sample_rows,pred_rows,diff
0,PXD004010,10,10,0
1,PXD016436,18,18,0
2,PXD019519,6,6,0
3,PXD025663,12,12,0
4,PXD040582,24,24,0
5,PXD050621,9,9,0
6,PXD061009,2,2,0
7,PXD061090,6,6,0
8,PXD061136,2,2,0
9,PXD061195,1376,1376,0


saved -> submission_v3.csv


Achieved a score of 0.15876 on the leaderboard which isn't great.